# TopoLines

Turn images into topographic contour lines via the eikonal equation:

1. Grayscale → normalize → slight Gaussian blur  
2. Build a speed field from intensity (`speed = ε + image`)  
3. Place seed point(s)  
4. Solve `|∇T| = 1/speed` with **scikit-fmm** (monotone fast marching)  
5. Draw iso-contours of the travel-time field `T` as thin lines  

In [ ]:
# If packages are missing, create/use the project venv then install:
#   python3.12 -m venv .venv
#   .venv/bin/pip install -r requirements.txt
#   .venv/bin/python -m ipykernel install --user --name topolines --display-name "TopoLines"
# Select kernel "TopoLines" (or this folder's .venv) in the notebook UI.
#
# %pip install scikit-fmm scikit-image matplotlib pillow numpy scipy ipywidgets ipympl

import skfmm  # noqa: F401 — fail early if the kernel is wrong
print("skfmm", getattr(skfmm, "__version__", "ok"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt; plt.close('all')  # clear any figures left from a previous session

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import skfmm
from PIL import Image
from scipy.ndimage import gaussian_filter
from skimage import measure

IMAGE_PATH = Path("examples/einstein.jpg")
BLUR_SIGMA = 1.0          # light blur so pixel noise doesn't spawn bogus wavefronts
SPEED_EPS = 0.1           # keep speed > 0 to avoid infinite travel time
STRIPE_SPACING = 8.0      # L — main tuning knob (smaller → denser stripes)
SEED_MODE = "center"      # "center" | "scatter" | "edge"
N_SCATTER = 5             # used when SEED_MODE == "scatter"
FIGSIZE = (10, 4)

## 1. Load, grayscale, normalize, blur

In [ ]:
def load_grayscale(path: Path) -> np.ndarray:
    """Load image as float intensity in [0, 1]."""
    rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.float64)
    # Luminance weights (Rec. 601)
    gray = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    gray = gray / 255.0
    return np.clip(gray, 0.0, 1.0)


def preprocess(image: np.ndarray, blur_sigma: float = BLUR_SIGMA) -> np.ndarray:
    """Normalize and lightly blur. Skip blur → noisy local wavefront junk."""
    img = image.astype(np.float64)
    lo, hi = img.min(), img.max()
    if hi > lo:
        img = (img - lo) / (hi - lo)
    if blur_sigma > 0:
        img = gaussian_filter(img, sigma=blur_sigma)
        img = np.clip(img, 0.0, 1.0)
    return img


raw = load_grayscale(IMAGE_PATH)
image = preprocess(raw)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
axes[0].imshow(raw, cmap="gray")
axes[0].set_title("grayscale")
axes[0].axis("off")
axes[1].imshow(image, cmap="gray")
axes[1].set_title(f"normalized + blur (σ={BLUR_SIGMA})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"shape={image.shape}, range=[{image.min():.3f}, {image.max():.3f}]")

## 2–3. Speed field + seed points

Bright pixels → faster wavefront travel. Seeds mark where the wave starts (`phi = -1`).

In [ ]:
def make_speed(image: np.ndarray, eps: float = SPEED_EPS) -> np.ndarray:
    """speed = eps + intensity; never allow 0 (→ ∞ travel time)."""
    return eps + image


def make_seeds(
    shape: tuple[int, int],
    mode: str = SEED_MODE,
    n_scatter: int = N_SCATTER,
    rng: np.random.Generator | None = None,
) -> list[tuple[int, int]]:
    """Return (row, col) seed locations.

    - center: radial expansion from image midpoint
    - scatter: a few random interior points
    - edge: seeds along the left edge → roughly parallel flow
    """
    h, w = shape
    if mode == "center":
        return [(h // 2, w // 2)]
    if mode == "scatter":
        rng = rng or np.random.default_rng(0)
        margin = max(8, min(h, w) // 16)
        ys = rng.integers(margin, h - margin, size=n_scatter)
        xs = rng.integers(margin, w - margin, size=n_scatter)
        return list(zip(ys.tolist(), xs.tolist()))
    if mode == "edge":
        # Dense seeds along left edge give nearly planar wavefronts
        step = max(1, h // 64)
        return [(y, 0) for y in range(0, h, step)]
    raise ValueError(f"unknown SEED_MODE: {mode!r}")


def make_phi(shape: tuple[int, int], seeds: list[tuple[int, int]]) -> np.ndarray:
    """Narrow-band mask for skfmm: -1 at seeds, +1 elsewhere."""
    phi = np.ones(shape, dtype=np.float64)
    for y, x in seeds:
        phi[y, x] = -1.0
    return phi


speed = make_speed(image)
seeds = make_seeds(image.shape, mode=SEED_MODE)
phi = make_phi(image.shape, seeds)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
im0 = axes[0].imshow(speed, cmap="magma")
axes[0].set_title(f"speed = {SPEED_EPS} + image")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

axes[1].imshow(image, cmap="gray")
sy, sx = zip(*seeds)
axes[1].scatter(sx, sy, c="cyan", s=36, marker="x", linewidths=1.5, zorder=3)
axes[1].set_title(f"seeds ({SEED_MODE}, n={len(seeds)})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Solve the eikonal equation with scikit-fmm

`skfmm.travel_time` is a proper upwind / fast-marching solver of `|∇T| = 1/speed`.

In [ ]:
def solve_eikonal(phi: np.ndarray, speed: np.ndarray) -> np.ndarray:
    """Travel-time field T from seeds through the speed medium."""
    return skfmm.travel_time(phi, speed)


T = solve_eikonal(phi, speed)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(T, cmap="viridis")
ax.set_title("travel time T")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046, label="T")
plt.tight_layout()
plt.show()

print(f"T range=[{T.min():.3f}, {T.max():.3f}]")

## 5. Stripes from iso-contours of T

Vector contours (cleanest) via `skimage.measure.find_contours`, plus an optional raster preview using `cos(2πT/L)`.

In [ ]:
def extract_contours(T: np.ndarray, L: float) -> list[np.ndarray]:
    """Iso-contours of T at levels 0, L, 2L, ..."""
    levels = np.arange(0.0, float(np.nanmax(T)), L)
    contours: list[np.ndarray] = []
    for level in levels:
        contours.extend(measure.find_contours(T, level))
    return contours


def plot_vector_stripes(
    T: np.ndarray,
    L: float,
    ax=None,
    color: str = "black",
    linewidth: float = 0.4,
    bg=None,
):
    """Draw thin contour lines of T — matches the classic Eikonal print look."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    if bg is not None:
        ax.imshow(bg, cmap="gray", alpha=0.25)
    else:
        ax.set_facecolor("white")
        ax.set_xlim(0, T.shape[1])
        ax.set_ylim(T.shape[0], 0)  # image coords: y down
    for contour in extract_contours(T, L):
        # find_contours returns (row, col) == (y, x)
        ax.plot(contour[:, 1], contour[:, 0], color=color, lw=linewidth, solid_capstyle="round")
    ax.set_aspect("equal")
    ax.axis("off")
    return ax


def raster_stripes(T: np.ndarray, L: float) -> np.ndarray:
    """Raster preview: cos(2πT/L). Threshold for hard stripes; softer midtones for AA-ish look."""
    return np.cos(2.0 * np.pi * T / L)


contours = extract_contours(T, STRIPE_SPACING)
raster = raster_stripes(T, STRIPE_SPACING)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

plot_vector_stripes(T, STRIPE_SPACING, ax=axes[0], linewidth=0.35)
axes[0].set_title(f"vector contours (L={STRIPE_SPACING})")

axes[1].imshow(raster > 0, cmap="gray")
axes[1].set_title(f"raster cos(2πT/L) thresholded (L={STRIPE_SPACING})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"{len(contours)} contour polylines at spacing L={STRIPE_SPACING}")

## End-to-end helper + seed-mode comparison

`eikonal_stripes` wraps the full pipeline. Re-run with different `seed_mode` / `L` to tune.

In [ ]:
def eikonal_stripes(
    path: Path | str,
    *,
    blur_sigma: float = BLUR_SIGMA,
    speed_eps: float = SPEED_EPS,
    seed_mode: str = SEED_MODE,
    n_scatter: int = N_SCATTER,
    L: float = STRIPE_SPACING,
    linewidth: float = 0.35,
    show: bool = True,
):
    """Full pipeline: image → T → vector stripe plot."""
    raw = load_grayscale(Path(path))
    img = preprocess(raw, blur_sigma=blur_sigma)
    spd = make_speed(img, eps=speed_eps)
    seeds = make_seeds(img.shape, mode=seed_mode, n_scatter=n_scatter)
    phi = make_phi(img.shape, seeds)
    travel = solve_eikonal(phi, spd)

    if show:
        h_img, w_img = img.shape
        _pw = 4.0
        fig, axes = plt.subplots(1, 3, figsize=(_pw * 3, _pw * h_img / w_img))
        axes[0].imshow(img, cmap="gray")
        sy, sx = zip(*seeds)
        axes[0].scatter(sx, sy, c="cyan", s=28, marker="x", linewidths=1.2)
        axes[0].set_title(f"input + seeds ({seed_mode})")
        axes[0].axis("off")

        im = axes[1].imshow(travel, cmap="viridis")
        axes[1].set_title("travel time T")
        axes[1].axis("off")
        plt.colorbar(im, ax=axes[1], fraction=0.046)

        plot_vector_stripes(travel, L, ax=axes[2], linewidth=linewidth)
        axes[2].set_title(f"eikonal stripes (L={L})")
        plt.tight_layout()
        plt.show()

    return travel, seeds


# Compare the three seed layouts side by side
h_img, w_img = image.shape
_pw = 4.0
fig, axes = plt.subplots(1, 3, figsize=(_pw * 3, _pw * h_img / w_img))
for ax, mode in zip(axes, ("center", "scatter", "edge")):
    travel, _ = eikonal_stripes(IMAGE_PATH, seed_mode=mode, L=STRIPE_SPACING, show=False)
    plot_vector_stripes(travel, STRIPE_SPACING, ax=ax, linewidth=0.3)
    ax.set_title(mode)
plt.suptitle(f"seed modes compared (L={STRIPE_SPACING})", y=1.02)
plt.tight_layout()
plt.show()

## Playground — tweak modifiers

Drag the sliders (or edit values) and watch the stripes update.  
`L` densifies/thins stripes; blur cleans noise; `ε` flattens contrast in the speed field; seed mode changes the whole flow.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display

# ── knobs ──────────────────────────────────────────────────────────────
# Re-run this cell after changing code above. Use the widgets below to explore.
w_blur = widgets.FloatSlider(
    value=BLUR_SIGMA, min=0.0, max=5.0, step=0.25,
    description="blur σ", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
    tooltip="Higher → smoother speed field, fewer noisy wiggles in stripes",
)
w_eps = widgets.FloatSlider(
    value=SPEED_EPS, min=0.01, max=1.0, step=0.01,
    description="speed ε", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
    tooltip="Floor on speed. Higher → more uniform travel, less image-driven distortion",
)
w_L = widgets.FloatSlider(
    value=STRIPE_SPACING, min=1.0, max=40.0, step=0.5,
    description="L (spacing)", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
    tooltip="Stripe period. Small → dense/noisy; large → sparse/abstract",
)
w_lw = widgets.FloatSlider(
    value=0.35, min=0.1, max=2.0, step=0.05,
    description="line width", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
)
w_seed = widgets.Dropdown(
    options=[("center (radial)", "center"), ("scatter", "scatter"), ("edge (parallel flow)", "edge")],
    value=SEED_MODE,
    description="seed mode",
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
)
w_n = widgets.IntSlider(
    value=N_SCATTER, min=1, max=20, step=1,
    description="# scatter", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
    tooltip="Only used when seed mode is scatter",
)
w_invert = widgets.Checkbox(
    value=False,
    description="invert speed (dark = fast)",
    indent=False,
    layout=widgets.Layout(width='auto'),
)
w_show_T = widgets.Checkbox(
    value=False,
    description="show travel-time field",
    indent=False,
    layout=widgets.Layout(width='auto'),
)
w_overlay = widgets.Checkbox(
    value=False,
    description="ghost original under stripes",
    indent=False,
    layout=widgets.Layout(width='auto'),
)

# Render into an Image widget (updated in place) instead of an Output widget.
# Appending to an Output via display() can accumulate stale outputs when the
# frontend fires extra value-sync events, which caused duplicated image pairs.
img_widget = widgets.Image(format="png")
stats = widgets.HTML()

_last_params = None  # skip redundant renders when frontend echoes the same values


def _render(_=None):
    global _last_params
    params = (
        w_blur.value, w_eps.value, w_L.value, w_lw.value,
        w_seed.value, w_n.value, w_invert.value, w_show_T.value, w_overlay.value,
    )
    if params == _last_params:
        return
    _last_params = params

    raw = load_grayscale(IMAGE_PATH)
    img = preprocess(raw, blur_sigma=w_blur.value)

    intensity = (1.0 - img) if w_invert.value else img
    spd = make_speed(intensity, eps=w_eps.value)
    seeds = make_seeds(img.shape, mode=w_seed.value, n_scatter=w_n.value)
    travel = solve_eikonal(make_phi(img.shape, seeds), spd)

    cols = 3 if w_show_T.value else 2
    h_img, w_img = img.shape
    panel_w = 4.0
    panel_h = panel_w * h_img / w_img  # match image aspect so set_aspect('equal') leaves no gap

    # Build figure completely off-screen using the Agg backend directly,
    # so no matplotlib display hook can auto-show it.
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(figsize=(panel_w * cols, panel_h))
    FigureCanvasAgg(fig)
    axes = list(np.atleast_1d(fig.subplots(1, cols)))

    axes[0].imshow(img, cmap="gray")
    sy, sx = zip(*seeds)
    axes[0].scatter(sx, sy, c="cyan", s=36, marker="x", linewidths=1.4, zorder=3)
    axes[0].set_title(f"input + seeds ({w_seed.value})")
    axes[0].axis("off")

    stripe_ax = axes[1]
    if w_show_T.value:
        im = axes[1].imshow(travel, cmap="viridis")
        axes[1].set_title("travel time T")
        axes[1].axis("off")
        fig.colorbar(im, ax=axes[1], fraction=0.046)
        stripe_ax = axes[2]

    bg = img if w_overlay.value else None
    plot_vector_stripes(travel, w_L.value, ax=stripe_ax, linewidth=w_lw.value, bg=bg)
    inv = ", inverted" if w_invert.value else ""
    stripe_ax.set_title(f"stripes  L={w_L.value:g}  σ={w_blur.value:g}  ε={w_eps.value:g}{inv}")

    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    img_widget.value = buf.getvalue()  # in-place update → always exactly one image
    del fig

    stats.value = (
        f"<code>T∈[{travel.min():.1f}, {travel.max():.1f}]  |  "
        f"speed∈[{spd.min():.3f}, {spd.max():.3f}]  |  "
        f"seeds={len(seeds)}</code>"
    )


controls = widgets.VBox([
    widgets.HTML("<b>Modifiers</b> — change any control; plot refreshes on release"),
    w_blur, w_eps, w_L, w_lw, w_seed, w_n,
    widgets.HBox([w_invert, w_show_T, w_overlay], layout=widgets.Layout(gap='20px', justify_content='flex-start')),
])

display(controls, img_widget, stats)
_render()

# Attach observers AFTER display + initial render so that widget frontend
# state-sync during display() can't trigger spurious re-renders.
for w in (w_blur, w_eps, w_L, w_lw, w_seed, w_n, w_invert, w_show_T, w_overlay):
    w.observe(_render, names="value")


## Click-to-seed playground

Same pipeline as above, but **left-click** the left panel to place seeds (as many as you want). **Right-click** undoes the last seed; **Clear** resets. Stripes refresh after each change.


In [ ]:
# Click-to-place seeds. Needs ipympl for matplotlib click events:
#   pip install ipympl
# Cursor/VS Code may prompt once to download the jupyter-matplotlib widget
# script from a CDN (jsdelivr/unpkg) — that's the JS frontend, not Python packages.
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output

# ── knobs (same modifiers as the playground above) ─────────────────────
cw_blur = widgets.FloatSlider(
    value=BLUR_SIGMA, min=0.0, max=5.0, step=0.25,
    description="blur σ", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
)
cw_eps = widgets.FloatSlider(
    value=SPEED_EPS, min=0.01, max=1.0, step=0.01,
    description="speed ε", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
)
cw_L = widgets.FloatSlider(
    value=STRIPE_SPACING, min=1.0, max=40.0, step=0.5,
    description="L (spacing)", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
)
cw_lw = widgets.FloatSlider(
    value=0.35, min=0.1, max=2.0, step=0.05,
    description="line width", continuous_update=False,
    style={"description_width": "90px"}, layout=widgets.Layout(width="420px"),
)
cw_invert = widgets.Checkbox(value=False, description="invert speed (dark = fast)", indent=False, layout=widgets.Layout(width='auto'))
cw_show_T = widgets.Checkbox(value=False, description="show travel-time field", indent=False, layout=widgets.Layout(width='auto'))
cw_overlay = widgets.Checkbox(value=False, description="ghost original under stripes", indent=False, layout=widgets.Layout(width='auto'))

btn_clear = widgets.Button(description="Clear seeds", button_style="warning")
btn_undo = widgets.Button(description="Undo last")
seed_status = widgets.HTML(value="<i>No seeds yet — click the left panel to place some.</i>")

click_seeds: list[tuple[int, int]] = []
_click_state: dict = {"shape": None, "cid": None, "fig": None}

click_out = widgets.Output()


def _set_seed_status():
    n = len(click_seeds)
    if n == 0:
        seed_status.value = "<i>No seeds yet — left-click the left panel to place some.</i>"
    else:
        pts = ", ".join(f"({y},{x})" for y, x in click_seeds[-5:])
        more = f" … +{n - 5} earlier" if n > 5 else ""
        seed_status.value = f"<b>{n} seed{'s' if n != 1 else ''}</b> (row, col): {pts}{more}"


def _render_click(_=None):
    with click_out:
        clear_output(wait=True)

        raw = load_grayscale(IMAGE_PATH)
        img = preprocess(raw, blur_sigma=cw_blur.value)
        _click_state["shape"] = img.shape

        intensity = (1.0 - img) if cw_invert.value else img
        spd = make_speed(intensity, eps=cw_eps.value)

        # Recreate figure with the right number of panels to avoid wasted whitespace
        cols = 3 if (cw_show_T.value and click_seeds) else 2
        h_img, w_img = img.shape
        panel_w = 4.0
        panel_h = panel_w * h_img / w_img  # match image aspect so set_aspect('equal') leaves no gap
        fig, axes_arr = plt.subplots(1, cols, figsize=(panel_w * cols, panel_h), dpi=100)
        axes = list(np.atleast_1d(axes_arr))

        # Disconnect old click handler then attach to new figure
        old_fig = _click_state.get("fig")
        old_cid = _click_state.get("cid")
        if old_fig is not None and old_cid is not None:
            try:
                old_fig.canvas.mpl_disconnect(old_cid)
            except Exception:
                pass
        _click_state["fig"] = fig
        _click_state["cid"] = fig.canvas.mpl_connect("button_press_event", _on_click)

        # Panel 0: image + seeds
        axes[0].imshow(img, cmap="gray")
        axes[0].set_title(f"click to place seeds ({len(click_seeds)})", fontsize=10)
        axes[0].axis("off")
        if click_seeds:
            sy, sx = zip(*click_seeds)
            axes[0].scatter(sx, sy, c="cyan", s=36, marker="x", linewidths=1.4, zorder=3)

        if not click_seeds:
            axes[1].imshow(img, cmap="gray", alpha=0.25)
            axes[1].text(
                0.5, 0.5, "add at least one seed\nto compute travel time",
                ha="center", va="center", transform=axes[1].transAxes,
                color="0.3", fontsize=9,
            )
            axes[1].set_title("stripes", fontsize=10)
            axes[1].axis("off")
            _set_seed_status()
            plt.tight_layout(pad=0.4)
            plt.show()
            return

        travel = solve_eikonal(make_phi(img.shape, click_seeds), spd)

        stripe_ax = axes[1]
        if cw_show_T.value and cols == 3:
            im = axes[1].imshow(travel, cmap="viridis")
            axes[1].set_title("travel time T", fontsize=10)
            axes[1].axis("off")
            fig.colorbar(im, ax=axes[1], fraction=0.046)
            stripe_ax = axes[2]

        bg = img if cw_overlay.value else None
        plot_vector_stripes(travel, cw_L.value, ax=stripe_ax, linewidth=cw_lw.value, bg=bg)
        inv = ", inverted" if cw_invert.value else ""
        stripe_ax.set_title(
            f"stripes  L={cw_L.value:g}  σ={cw_blur.value:g}  ε={cw_eps.value:g}{inv}",
            fontsize=10,
        )
        stripe_ax.axis("off")

        _set_seed_status()
        plt.tight_layout(pad=0.4)
        plt.show()


def _on_click(event):
    fig = _click_state.get("fig")
    axes = getattr(fig, "_click_axes", None)
    # Identify the input panel by checking the first axis of the current figure
    if fig is None or event.inaxes is None:
        return
    # Only respond to clicks in the first (leftmost) axes of the current figure
    if event.inaxes is not fig.axes[0]:
        return
    shape = _click_state["shape"]
    if shape is None or event.xdata is None or event.ydata is None:
        return
    h, w = shape
    r = int(np.clip(round(event.ydata), 0, h - 1))
    c = int(np.clip(round(event.xdata), 0, w - 1))
    if event.button == 1:  # left-click → add
        click_seeds.append((r, c))
    elif event.button == 3:  # right-click → undo
        if not click_seeds:
            return
        click_seeds.pop()
    else:
        return
    _render_click()


def _clear_seeds(_=None):
    click_seeds.clear()
    _render_click()


def _undo_seed(_=None):
    if click_seeds:
        click_seeds.pop()
        _render_click()


btn_clear.on_click(_clear_seeds)
btn_undo.on_click(_undo_seed)

click_controls = widgets.VBox([
    widgets.HTML(
        "<b>Click-to-seed</b> — left-click left panel to add; right-click to undo; "
        "sliders refresh on release"
    ),
    cw_blur, cw_eps, cw_L, cw_lw,
    widgets.HBox([cw_invert, cw_show_T, cw_overlay], layout=widgets.Layout(gap='20px', justify_content='flex-start')),
    widgets.HBox([btn_clear, btn_undo]),
    seed_status,
])
for w in (cw_blur, cw_eps, cw_L, cw_lw, cw_invert, cw_show_T, cw_overlay):
    w.observe(_render_click, names="value")

display(click_controls, click_out)
_render_click()
